In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 01 - Environment test\n",
    "\n",
    "Phase 2 verification notebook for the project\n",
    "\n",
    "**Adaptive CUDA Thread-Block Configuration for Workload-Aware GPU Optimization**\n",
    "\n",
    "This notebook is designed for **Google Colab** (or any Linux GPU host).\n",
    "It does **not** benchmark anything.  Its only job is to confirm that the\n",
    "GPU, CUDA toolkit, CMake toolchain, and the project's own\n",
    "`device_query` / `gpu_info` probes all work end to end.\n",
    "\n",
    "Run the cells in order."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 1: Python version ---------------------------------------------\n",
    "import sys, platform\n",
    "print('Python :', sys.version)\n",
    "print('Platform:', platform.platform())\n",
    "print('Machine :', platform.machine())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 2: NVIDIA GPU present? ---------------------------------------\n",
    "!nvidia-smi"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 3: CUDA toolkit (nvcc) ---------------------------------------\n",
    "!which nvcc || echo 'nvcc not on PATH'\n",
    "!nvcc --version || echo 'nvcc --version failed'"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 4: CMake version ---------------------------------------------\n",
    "!cmake --version"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 5: Locate or clone the repository ----------------------------\n",
    "import os, pathlib\n",
    "\n",
    "REPO_URL = 'https://github.com/YOUR-USER/YOUR-REPO.git'  # <-- set this\n",
    "REPO_DIR = pathlib.Path('/content/adaptive-cuda-thread-block')\n",
    "\n",
    "if (REPO_DIR / 'CMakeLists.txt').exists():\n",
    "    print('Repository already present at', REPO_DIR)\n",
    "elif pathlib.Path('CMakeLists.txt').exists():\n",
    "    REPO_DIR = pathlib.Path.cwd()\n",
    "    print('Repository is the current directory:', REPO_DIR)\n",
    "else:\n",
    "    !git clone $REPO_URL $REPO_DIR\n",
    "    print('Cloned into', REPO_DIR)\n",
    "\n",
    "os.chdir(REPO_DIR)\n",
    "print('cwd =', pathlib.Path.cwd())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 6: Install Python requirements --------------------------------\n",
    "!python -m pip install --quiet --upgrade pip\n",
    "!python -m pip install --quiet -r requirements.txt\n",
    "print('Python dependencies installed.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 7: Configure and build the CUDA targets -----------------------\n",
    "!rm -rf build\n",
    "!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release\n",
    "!cmake --build build --config Release --parallel"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 8: Run the CUDA device query ---------------------------------\n",
    "import pathlib, subprocess, sys\n",
    "\n",
    "binary = pathlib.Path('build/bin/device_query')\n",
    "if not binary.exists():\n",
    "    raise SystemExit(f'device_query not found at {binary.resolve()}')\n",
    "\n",
    "print('--- human-readable ---')\n",
    "print(subprocess.run([str(binary)], capture_output=True, text=True).stdout)\n",
    "\n",
    "print('--- json ---')\n",
    "print(subprocess.run([str(binary), '--json'], capture_output=True, text=True).stdout)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 9: Run the Python GPU environment report ---------------------\n",
    "!python -m src.profiling.gpu_info\n",
    "!python -m src.profiling.gpu_info --json"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 10: Run the project test suite -------------------------------\n",
    "!python -m pytest -q tests/"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Cell 11: Verdict --------------------------------------------------\n",
    "import json, subprocess\n",
    "out = subprocess.run([sys.executable, '-m', 'src.profiling.gpu_info', '--json'],\n",
    "                     capture_output=True, text=True).stdout\n",
    "env = json.loads(out)\n",
    "print('gpu_ready =', env['gpu_ready'])\n",
    "if env['gpu_ready']:\n",
    "    print('Environment is READY for Phase 3.')\n",
    "else:\n",
    "    print('Environment is NOT ready.  Inspect the report above.')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}